这是一份根据我们多次深入探讨整理而成的**《A股日内做T时序模型：从直觉到落地全流程设计文档》**。

这份文档旨在为你和老板开会提供逻辑支持，同时也作为你后续编写代码的工程指南。

---

# A股日内做T时序模型设计方案：基于能量偏差与动态VWAP预测

## 1. 核心背景与业务目标
*   **背景**：在已有“Top 100 截面选股模型”（T+2 持仓）的基础上，利用日内波动进行仓位置换（做T）。
*   **目标**：不改变选股池，通过“高卖低买”降低持仓成本，获取日内执行 Alpha。
*   **核心挑战**：覆盖 A 股约 **0.15% - 0.2%** 的摩擦成本（印花税、佣金、滑点）。

---

## 2. 核心直觉：均值回归与相对价值
模型的底层逻辑建立在两个市场观察之上：

### 2.1 能量偏差引力 (The $X_2$ Intuition)
*   **VWAP (资金成本线)** vs **TWAP (时间平衡线)**。
*   当 $VWAP > TWAP$ 时，说明大成交量发生在高位，若价格随后无力支撑，则存在向资金中枢回归的强烈拉力。
*   **测试结论**：$X_2$（VWAP/TWAP偏差）是比单纯的价格偏差（$X_1$）更稳健的回归信号。

### 2.2 动态相对廉价度 (Boss's Idea)
*   **5分钟 VWAP ($V_{5m}$)**：代表你当下入场的即时成本。
*   **剩余时间 VWAP ($V_{rest}$)**：代表从现在起到收盘的预期平均成本。
*   **逻辑**：如果能预测出接下来的 5 分钟价格是今天剩下的时间里最便宜的（$V_{5m} < V_{rest}$），那么此时买入就是绝对正确的。

---

## 3. 特征工程：量化“橡皮筋”的张力

为了识别机会，我们需要构建以下特征（建议使用 **Z-Score 标准化** 处理，以适配不同波动率的股票）：

| 特征代码 | 名称 | 计算公式 | 业务逻辑 |
| :--- | :--- | :--- | :--- |
| **$X_1$** | 价格偏离度 | `Price / VWAP - 1` | 价格脱离当前成本中枢的程度 |
| **$X_2$** | 能量偏离度 | `VWAP / TWAP - 1` | 资金成交节奏的偏差（核心因子） |
| **$Z_{final}$** | 组合因子 | `0.5*Z(X1) + 2.5*Z(X2)` | 强化稳健的能量信号，压制不稳定的价格动量 |
| **$X_{time}$** | 时间因子 | `Time / 15:00` | 越接近收盘，均值回归的确定性越高 |
| **$X_{vol}$** | 相对成交量 | `Vol_5m / Avg_Vol_5m_hist` | 放量通常是日内趋势反转的信号 |

---

## 4. 标签设计：预测“套利空间”

老板提出的 5 分钟滚动扫描思路，要求我们将 Label ($Y$) 定义为一个**比例关系**。

### 4.1 标签公式
$$Y_t = \frac{VWAP_{(t \to t+5min)}}{VWAP_{(t+5min \to 15:00)}}$$

### 4.2 阈值定义（三分类模型）
*   **Label = 1 (买入信号)**：当 $Y_t < 0.995$。
    *   *解释*：未来5分钟价格比剩余时间均价便宜 0.5% 以上。
*   **Label = -1 (卖出信号)**：当 $Y_t > 1.005$。
    *   *解释*：未来5分钟价格比剩余时间均价贵 0.5% 以上。
*   **Label = 0 (观望)**：波动不足以覆盖手续费。

---

## 5. 从 0 到 1 的工程实施步骤

### 第一步：历史数据切片 (Data Preparation)
1.  提取中证 2000 等高波动个股的 **1分钟线** 数据。
2.  将全天 240 分钟划分为 48 个 **5分钟 Bar**。
3.  对每个 Bar 计算其对应的 $V_{5m}$ 和 $V_{rest}$（这是回测时的“上帝视角”标签）。

### 第二步：标准化特征计算 (Feature Scaling)
1.  对 $X_1, X_2$ 计算过去 20 个交易日的滚动均值和标准差。
2.  将特征转化为 **Z-Score**，确保平安银行和通达股份在同一个维度被比较。

### 第三步：训练滚动模型 (Model Training)
1.  使用 **LightGBM** 或 **XGBoost**。
2.  **输入**：$t$ 时刻及其之前的时序特征。
3.  **目标**：预测该时刻的分类标签（1, -1, 0）。

### 第四步：执行策略设计 (Execution)
*   **触发条件**：$Z_{final}$ 达到极值（如 $> 1.5$ 或 $< -1.5$）且模型预测 Label 为 1 或 -1。
*   **离场逻辑**：
    *   方案 A：持有至 14:50 强制平仓。
    *   方案 B（动态）：当 $Z_{final}$ 回归到 0 附近时提前平仓，提高资金周转率。

---

## 6. 预期收益归因与风险预警

### 6.1 预期收益来源 (Alpha Source)
*   **小盘股弹性**：中证 2000 标的的标准差普遍在 1% 以上，提供了足够的获利空间。
*   **共振效应**：当 $X_1, X_2$ 同时偏离超过 1 个标准差时，回归的平均空间可达 **0.63%** 以上，远超 **0.2%** 的成本线。
*   **频率增益**：每 5 分钟扫描一次，大大增加了捕获瞬时极端偏差的机会。

### 6.2 潜在风险 (Risk Warnings)
*   **单边市风险**：在极强的动量行情中（如单边暴跌），$X_1$ 的正相关性（动量）会抵消 $X_2$ 的回归力，导致“接飞刀”。
    *   *对策*：引入大盘指数作为滤网。
*   **手续费磨损**：如果预测精度不足，频繁交易会导致本金被手续费蚕食。
    *   *对策*：严控阈值，只做“高确定性”的极端偏离。

---

## 7. 结语：程序员的落地建议
建议第一步先在代码中实现 **`remaining_vwap`** 的计算逻辑，并统计它与 **`X2_zscore`** 的相关性。如果全样本下两者的相关性稳定且为负，那么老板的“5分钟相对价值”模型就具备了实盘的基础。